# Day 2 Lab — Solutions

**Name:** Walaa Omar Hassan  
**Lab:** Build an AI Therapist RAG System (Manual + LangGraph)  


---


# 🧠 Day 2 Lab: Build an AI Therapist RAG System

**Welcome back!** Today we're building something real — an AI therapy assistant powered by RAG.

We'll do it **twice**:
- **Part A**: The manual way (pure Python, no frameworks) — so you understand every step
- **Part B**: The smart way (LangGraph + LangSmith) — so you see how production systems work

### What you'll build:
A system that answers therapy-related questions using CBT (Cognitive Behavioral Therapy) documents, with:
- 🚨 Crisis detection (keyword-based, no LLM needed!)
- 🔍 Smart retrieval with reranking
- 📝 Grounded answers with citations
- 📊 Relevance score thresholds

### Rules:
- Instructions tell you WHAT to do — YOU write the code
- If stuck > 5 min, ask!
- This is NOT an exam. Google is your friend.

---

## 🔧 Setup

In [1]:
!pip install -q langchain-text-splitters sentence-transformers chromadb numpy scikit-learn langgraph langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

---

## 📚 The Therapy Knowledge Base

Here's our therapy document. Just run this cell — don't change it.

Imagine these are chunks from real CBT manuals, clinical guidelines, and coping strategy databases.

In [2]:
THERAPY_DOCUMENTS = [
    {
        "text": "Cognitive Behavioral Therapy (CBT) is a structured, time-limited psychotherapy that aims to solve current problems by changing unhelpful thinking patterns and behaviors. CBT is based on the cognitive model: the way we perceive situations influences how we feel emotionally. It is not the situation itself that determines what people feel, but rather the way they interpret the situation. A typical CBT course lasts 12 to 20 sessions, with each session lasting about 50 minutes.",
        "metadata": {"source": "CBT Fundamentals Manual", "chapter": "Introduction to CBT", "page": 1, "category": "cbt_basics"}
    },
    {
        "text": "The CBT Triangle (also called the Cognitive Triangle) shows the connection between Thoughts, Feelings, and Behaviors. When you have a negative thought like 'I'm going to fail this exam', it leads to feelings of anxiety and dread, which leads to behaviors like avoiding study or procrastinating. CBT works by identifying and challenging these negative automatic thoughts to break the cycle.",
        "metadata": {"source": "CBT Fundamentals Manual", "chapter": "The Cognitive Triangle", "page": 5, "category": "cbt_basics"}
    },
    {
        "text": "Cognitive distortions are systematic errors in thinking that reinforce negative thought patterns. Common cognitive distortions include: (1) All-or-Nothing Thinking: seeing things in black and white, (2) Catastrophizing: expecting the worst possible outcome, (3) Mind Reading: assuming you know what others think, (4) Overgeneralization: making broad conclusions from a single event, (5) Emotional Reasoning: believing something is true because it feels true, (6) Should Statements: using 'should', 'must', 'ought to' rigidly.",
        "metadata": {"source": "CBT Fundamentals Manual", "chapter": "Cognitive Distortions", "page": 12, "category": "cbt_techniques"}
    },
    {
        "text": "Thought Records are a core CBT tool for examining and challenging negative thoughts. The 7-column thought record includes: (1) Situation: What happened? (2) Automatic Thought: What went through your mind? (3) Emotions: What did you feel? Rate intensity 0-100. (4) Evidence For: What supports this thought? (5) Evidence Against: What contradicts this thought? (6) Balanced Thought: A more realistic perspective. (7) Re-rate Emotions: How do you feel now? This structured approach helps patients develop more balanced thinking.",
        "metadata": {"source": "CBT Workbook", "chapter": "Thought Records", "page": 23, "category": "cbt_techniques"}
    },
    {
        "text": "The 5-4-3-2-1 Grounding Technique is used to manage acute anxiety and panic attacks. The patient identifies: 5 things they can SEE, 4 things they can TOUCH, 3 things they can HEAR, 2 things they can SMELL, 1 thing they can TASTE. This technique works by redirecting attention from anxious thoughts to the present moment through sensory engagement. It can be done anywhere and requires no special equipment.",
        "metadata": {"source": "Anxiety Management Guide", "chapter": "Grounding Techniques", "page": 8, "category": "anxiety"}
    },
    {
        "text": "Progressive Muscle Relaxation (PMR) involves systematically tensing and relaxing different muscle groups to reduce physical tension associated with anxiety. Start with the feet: tense the muscles for 5 seconds, then release for 30 seconds. Move upward through calves, thighs, abdomen, chest, hands, arms, shoulders, neck, and face. A full PMR session takes about 15-20 minutes. Regular practice (daily for 2 weeks) significantly reduces baseline anxiety levels.",
        "metadata": {"source": "Anxiety Management Guide", "chapter": "Relaxation Techniques", "page": 15, "category": "anxiety"}
    },
    {
        "text": "Behavioral Activation is a key treatment for depression. When people are depressed, they tend to withdraw from activities they used to enjoy, which makes depression worse (the depression cycle). Behavioral Activation breaks this cycle by scheduling pleasurable and meaningful activities, even when motivation is low. Start small: a 10-minute walk, calling a friend, or cooking a simple meal. Track mood before and after each activity to demonstrate the connection between action and mood improvement.",
        "metadata": {"source": "Depression Treatment Protocol", "chapter": "Behavioral Activation", "page": 7, "category": "depression"}
    },
    {
        "text": "Sleep hygiene is critically important for mental health. Poor sleep worsens both anxiety and depression. Key sleep hygiene practices include: maintain a consistent sleep schedule (same bedtime and wake time daily), avoid screens for 1 hour before bed, keep the bedroom cool and dark, avoid caffeine after 2 PM, exercise regularly but not within 3 hours of bedtime, use the bed only for sleep (not work or scrolling), and if you can't sleep after 20 minutes, get up and do something calming until sleepy.",
        "metadata": {"source": "Depression Treatment Protocol", "chapter": "Sleep Hygiene", "page": 18, "category": "depression"}
    },
    {
        "text": "Stress management through time management and boundaries is essential. The Eisenhower Matrix helps prioritize tasks: Urgent+Important (do now), Important+Not Urgent (schedule), Urgent+Not Important (delegate), Not Urgent+Not Important (eliminate). Setting boundaries means learning to say no to excessive demands, communicating limits clearly, and protecting time for self-care. Chronic stress without management can lead to burnout, anxiety disorders, and depression.",
        "metadata": {"source": "Stress Management Handbook", "chapter": "Time Management", "page": 5, "category": "stress"}
    },
    {
        "text": "Mindfulness meditation involves paying attention to the present moment without judgment. A simple practice: sit comfortably, close your eyes, focus on your breath. When your mind wanders (and it will), gently bring attention back to breathing without criticizing yourself. Start with 5 minutes daily, gradually increasing to 15-20 minutes. Research shows that 8 weeks of regular mindfulness practice reduces anxiety by 30-40% and improves emotional regulation.",
        "metadata": {"source": "Stress Management Handbook", "chapter": "Mindfulness", "page": 12, "category": "stress"}
    },
    {
        "text": "Exposure therapy is the gold standard treatment for phobias and anxiety disorders. It involves gradually and systematically confronting feared situations in a safe, controlled environment. The exposure hierarchy: list feared situations from least scary (anxiety rating 10/100) to most scary (anxiety rating 100/100). Start with the least scary and move up only when anxiety decreases to manageable levels. Never skip levels. Flooding (immediate full exposure) is generally not recommended.",
        "metadata": {"source": "Anxiety Management Guide", "chapter": "Exposure Therapy", "page": 22, "category": "anxiety"}
    },
    {
        "text": "Journaling for mental health: Expressive writing has been shown to reduce symptoms of anxiety and depression. Guidelines: write for 15-20 minutes about your thoughts and feelings. Don't worry about grammar or spelling. Focus on emotional expression rather than narrating events. Gratitude journaling (writing 3 things you're grateful for each day) has been shown to improve mood within 2 weeks. Combine with thought records for maximum benefit in CBT treatment.",
        "metadata": {"source": "CBT Workbook", "chapter": "Journaling Exercises", "page": 35, "category": "cbt_techniques"}
    },
    {
        "text": "CRISIS PROTOCOL: If a patient expresses suicidal thoughts, self-harm intentions, or is in immediate danger, do NOT attempt therapy. Instead: (1) Take it seriously, every time. (2) Ask directly: 'Are you thinking of hurting yourself?' (3) Listen without judgment. (4) Provide emergency contacts: National Suicide Prevention Lifeline: 988 (US), Crisis Text Line: text HOME to 741741. (5) Do not leave the person alone if risk is imminent. (6) Contact emergency services (911) if there is immediate danger. Safety always comes first.",
        "metadata": {"source": "Crisis Intervention Protocol", "chapter": "Suicide Risk", "page": 1, "category": "crisis", "priority": "HIGH"}
    }
]

print(f"Loaded {len(THERAPY_DOCUMENTS)} therapy documents")
for i, doc in enumerate(THERAPY_DOCUMENTS):
    print(f"  {i+1}. [{doc['metadata']['category']}] {doc['metadata']['chapter']} ({doc['metadata']['source']})")

Loaded 13 therapy documents
  1. [cbt_basics] Introduction to CBT (CBT Fundamentals Manual)
  2. [cbt_basics] The Cognitive Triangle (CBT Fundamentals Manual)
  3. [cbt_techniques] Cognitive Distortions (CBT Fundamentals Manual)
  4. [cbt_techniques] Thought Records (CBT Workbook)
  5. [anxiety] Grounding Techniques (Anxiety Management Guide)
  6. [anxiety] Relaxation Techniques (Anxiety Management Guide)
  7. [depression] Behavioral Activation (Depression Treatment Protocol)
  8. [depression] Sleep Hygiene (Depression Treatment Protocol)
  9. [stress] Time Management (Stress Management Handbook)
  10. [stress] Mindfulness (Stress Management Handbook)
  11. [anxiety] Exposure Therapy (Anxiety Management Guide)
  12. [cbt_techniques] Journaling Exercises (CBT Workbook)
  13. [crisis] Suicide Risk (Crisis Intervention Protocol)


---

# 🎯 PART A: Manual RAG Pipeline (No Frameworks)

We're going to build the entire pipeline by hand first. No magic, no frameworks hiding stuff from you. Just Python, embeddings, and logic.

---

## A1 — Crisis Detection (No LLM Needed! FREE!)

Before ANY retrieval or LLM call, check if the user is in crisis.

This is just keyword matching — zero cost, instant, potentially life-saving.

Write a function `check_crisis(query)` that:
1. Checks if the query contains any crisis keywords (suicide, kill myself, self-harm, end my life, want to die, hurt myself, etc.)
2. Returns `True` if crisis detected, `False` otherwise
3. If crisis detected, print the crisis response with emergency contacts

Test it with:
- `"I want to hurt myself"` → should trigger crisis
- `"I feel anxious about my exam"` → should NOT trigger crisis

In [3]:
# A1 — Crisis detection: pure keyword matching, zero cost, runs before anything else
import re

CRISIS_KEYWORDS = [
    # suicidal ideation
    "suicide", "suicidal", "kill myself", "killing myself", "take my own life",
    "end my life", "ending my life", "end it all", "want to die", "wanna die",
    "better off dead", "no reason to live", "don't want to live", "dont want to live",
    "can't go on", "cant go on", "nothing to live for",
    # self-harm
    "self harm", "self-harm", "hurt myself", "harm myself", "hurting myself",
    "cut myself", "cutting myself", "overdose", "kill me",
]

CRISIS_RESPONSE = """
================================================================
  I'm really glad you told me this. Your safety comes first.
================================================================

I'm an AI and I can't give you the help you deserve right now,
but there are people who can — 24/7, free, and confidential.

  United States
    * Call or text 988  (Suicide & Crisis Lifeline)
    * Text HOME to 741741  (Crisis Text Line)
    * Call 911 if you are in immediate danger

  Egypt
    * 08008880700  (Ministry of Health mental health hotline)
    * 0220816831   (Befrienders Cairo)
    * 123          (ambulance)

  Anywhere else
    * findahelpline.com lists a free helpline for almost every country

If you can, please don't stay alone right now. Reaching out to
someone you trust — a friend, a family member, a doctor — counts.

================================================================
"""


def check_crisis(query, verbose=True):
    """Check if user message indicates a crisis. Returns True if crisis detected."""
    text = query.lower()

    # \b word boundaries so "suicide" matches but "suicidepreventionmonth" as a
    # substring of a longer token doesn't produce weird partial hits
    matched = [
        kw for kw in CRISIS_KEYWORDS
        if re.search(r"\b" + re.escape(kw) + r"\b", text)
    ]

    if matched:
        if verbose:
            print(f"[CRISIS DETECTED] matched: {matched}")
            print(CRISIS_RESPONSE)
        return True
    return False


# Test it:
print(">>> Test 1:")
print("Result:", check_crisis("I want to hurt myself"))
print("\n>>> Test 2:")
print("Result:", check_crisis("I feel anxious about my exam"))

# A few more, to see the edges of a keyword-only approach
print("\n>>> Extra cases (verbose off):")
extra = [
    "I want to end my life",                       # True  - correct
    "Sometimes I think about suicide",             # True  - correct
    "I'm so tired I could die",                    # False - correct (idiom)
    "My friend attempted suicide last year",       # True  - FALSE POSITIVE
    "I don't see the point in anything anymore",   # False - FALSE NEGATIVE
]
for q in extra:
    print(f"  {str(check_crisis(q, verbose=False)):>5}  |  {q}")

print("""
Note on the two errors above:
  - False positive ("my friend attempted suicide"): acceptable. Showing a
    hotline to someone who didn't need it costs nothing.
  - False negative ("I don't see the point in anything anymore"): this is the
    dangerous one. Keywords cannot catch indirect phrasing. In production you
    layer a small classifier on top of the keyword list - the keywords stay
    because they are instant and free, and they catch the explicit cases.
  - Design rule: this check runs FIRST, before retrieval and before any LLM
    call, so a crisis can never depend on a model behaving well.
""")


>>> Test 1:
[CRISIS DETECTED] matched: ['hurt myself']

  I'm really glad you told me this. Your safety comes first.

I'm an AI and I can't give you the help you deserve right now,
but there are people who can — 24/7, free, and confidential.

  United States
    * Call or text 988  (Suicide & Crisis Lifeline)
    * Text HOME to 741741  (Crisis Text Line)
    * Call 911 if you are in immediate danger

  Egypt
    * 08008880700  (Ministry of Health mental health hotline)
    * 0220816831   (Befrienders Cairo)
    * 123          (ambulance)

  Anywhere else
    * findahelpline.com lists a free helpline for almost every country

If you can, please don't stay alone right now. Reaching out to
someone you trust — a friend, a family member, a doctor — counts.


Result: True

>>> Test 2:
Result: False

>>> Extra cases (verbose off):
   True  |  I want to end my life
   True  |  Sometimes I think about suicide
  False  |  I'm so tired I could die
   True  |  My friend attempted suicide last year

## A2 — Embeddings & Vector Store

1. Load the `all-MiniLM-L6-v2` embedding model
2. Create a ChromaDB collection called `"therapy_kb"`
3. Add all 13 documents with their text, metadata, and IDs
4. Print the collection count to verify

Remember: ChromaDB can handle embedding automatically if you set it up, OR you can embed yourself and pass embeddings. Either way works!

In [4]:
# A2 — Embeddings + vector store
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

# 2. Prepare the data
texts = [d["text"] for d in THERAPY_DOCUMENTS]
metadatas = [d["metadata"] for d in THERAPY_DOCUMENTS]
ids = [f"doc_{i}" for i in range(len(THERAPY_DOCUMENTS))]

# Chroma metadata values must be str/int/float/bool — our dicts already are,
# but normalise defensively (the crisis doc has an extra "priority" key)
metadatas = [
    {k: (v if isinstance(v, (str, int, float, bool)) else str(v)) for k, v in m.items()}
    for m in metadatas
]

# 3. Embed the documents ourselves (so retrieval and reranking use the SAME model)
doc_embeddings = model.encode(texts, show_progress_bar=True)
print("Embeddings matrix:", doc_embeddings.shape)

# 4. Create the collection
client = chromadb.Client()  # in-memory
try:
    client.delete_collection("therapy_kb")   # makes the cell safe to re-run
except Exception:
    pass

collection = client.create_collection(
    name="therapy_kb",
    metadata={"hnsw:space": "cosine"},   # cosine, not the L2 default
)

collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=doc_embeddings.tolist(),
)

print("\nDocuments in collection:", collection.count())
print("Sample metadata:", collection.peek(limit=2)["metadatas"])
print("""
Why cosine space matters here: with "hnsw:space": "cosine",
    distance = 1 - cosine_similarity
so distance lives in [0, 2], 0 = identical meaning, 1 = unrelated.
That is what makes the threshold in A4 interpretable. With the default L2
space the numbers would be unbounded and a fixed threshold meaningless.
""")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_767/3176743643.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings matrix: (13, 384)

Documents in collection: 13
Sample metadata: [{'chapter': 'Introduction to CBT', 'source': 'CBT Fundamentals Manual', 'page': 1, 'category': 'cbt_basics'}, {'category': 'cbt_basics', 'page': 5, 'chapter': 'The Cognitive Triangle', 'source': 'CBT Fundamentals Manual'}]

Why cosine space matters here: with "hnsw:space": "cosine",
    distance = 1 - cosine_similarity
so distance lives in [0, 2], 0 = identical meaning, 1 = unrelated.
That is what makes the threshold in A4 interpretable. With the default L2
space the numbers would be unbounded and a fixed threshold meaningless.



## A3 — Retrieval with Relevance Scores

Write a function `retrieve(query, top_k=5)` that:

1. Searches the ChromaDB collection
2. Returns the top-K results with their **relevance scores**
3. Prints each result with: rank, score, source, chapter, and first 100 chars of text

Then test with these queries:
- `"How do I deal with anxiety?"`
- `"What is the cognitive triangle?"`
- `"How can I sleep better?"`

Look at the relevance scores — are the top results actually relevant?

In [5]:
# A3 — Retrieval with relevance scores
def retrieve(query, collection, top_k=5, verbose=True):
    """Retrieve top-K relevant chunks with scores."""
    query_embedding = model.encode(query).reshape(1, -1).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    # Chroma returns a list-per-query; we only sent one, so flatten it
    flat = {
        "query": query,
        "documents": results["documents"][0],
        "metadatas": results["metadatas"][0],
        "distances": results["distances"][0],
    }

    if verbose:
        print("=" * 72)
        print(f"QUERY: {query}")
        print("=" * 72)
        for rank, (doc, meta, dist) in enumerate(
            zip(flat["documents"], flat["metadatas"], flat["distances"]), start=1
        ):
            print(f"[{rank}] distance={dist:.4f}  similarity={1 - dist:.4f}")
            print(f"    source : {meta['source']}")
            print(f"    chapter: {meta['chapter']} (page {meta['page']}, {meta['category']})")
            print(f"    text   : {doc[:100]}...")
            print()

    return flat


# Test queries:
for q in ["How do I deal with anxiety?",
          "What is the cognitive triangle?",
          "How can I sleep better?"]:
    retrieve(q, collection, top_k=3)


QUERY: How do I deal with anxiety?
[1] distance=0.4097  similarity=0.5903
    source : Stress Management Handbook
    chapter: Mindfulness (page 12, stress)
    text   : Mindfulness meditation involves paying attention to the present moment without judgment. A simple pr...

[2] distance=0.4984  similarity=0.5016
    source : Anxiety Management Guide
    chapter: Exposure Therapy (page 22, anxiety)
    text   : Exposure therapy is the gold standard treatment for phobias and anxiety disorders. It involves gradu...

[3] distance=0.5890  similarity=0.4110
    source : Depression Treatment Protocol
    chapter: Sleep Hygiene (page 18, depression)
    text   : Sleep hygiene is critically important for mental health. Poor sleep worsens both anxiety and depress...

QUERY: What is the cognitive triangle?
[1] distance=0.3631  similarity=0.6369
    source : CBT Fundamentals Manual
    chapter: The Cognitive Triangle (page 5, cbt_basics)
    text   : The CBT Triangle (also called the Cognitive Tri

## A4 — Relevance Score Threshold

Not all retrieved chunks are worth sending to the LLM!

Write a function `filter_by_relevance(results, threshold=1.0)` that:

1. Takes the retrieval results from A3
2. Filters OUT any chunk with distance > threshold (remember: in ChromaDB, **lower distance = more relevant**)
3. If ALL chunks are filtered out, return a message: `"I don't have enough information about that topic."`

Test with:
- `"What is exposure therapy?"` — should find relevant chunks
- `"What's the weather today?"` — should trigger "I don't have info" (out of scope!)

In [6]:
# A4 — Relevance threshold
def filter_by_relevance(results, threshold=1.0, verbose=True):
    """Filter results by relevance threshold. Lower distance = more relevant."""
    kept_docs, kept_metas, kept_dists = [], [], []

    for doc, meta, dist in zip(results["documents"], results["metadatas"], results["distances"]):
        if dist <= threshold:
            kept_docs.append(doc)
            kept_metas.append(meta)
            kept_dists.append(dist)

    if not kept_docs:
        message = "I don't have enough information about that topic."
        if verbose:
            print(message)
        return {"ok": False, "documents": [], "metadatas": [],
                "distances": [], "message": message}

    if verbose:
        print(f"Kept {len(kept_docs)}/{len(results['documents'])} chunks "
              f"(threshold = {threshold})")

    return {"ok": True, "documents": kept_docs, "metadatas": kept_metas,
            "distances": kept_dists, "message": None}


# --- Test the two cases --------------------------------------------------
print("#" * 72)
print('IN SCOPE: "What is exposure therapy?"')
print("#" * 72)
r_in = retrieve("What is exposure therapy?", collection, top_k=5, verbose=False)
print("distances:", [round(d, 3) for d in r_in["distances"]])
f_in = filter_by_relevance(r_in, threshold=1.0)
print("top chunk:", f_in["documents"][0][:120] if f_in["ok"] else "-", "...\n")

print("#" * 72)
print('OUT OF SCOPE: "What\'s the weather today?"')
print("#" * 72)
r_out = retrieve("What's the weather today?", collection, top_k=5, verbose=False)
print("distances:", [round(d, 3) for d in r_out["distances"]])
f_out = filter_by_relevance(r_out, threshold=1.0)


# --- Calibrating the threshold instead of guessing it --------------------
print("\n" + "#" * 72)
print("CALIBRATION — is 1.0 actually the right threshold?")
print("#" * 72)

# Include every in-scope query we will actually test later, so the threshold
# is guaranteed not to reject a question the system is supposed to answer.
in_scope_queries = [
    "What is exposure therapy?",
    "How do I challenge negative thoughts?",
    "What is the 5-4-3-2-1 technique?",
    "I can't sleep at night, any tips?",
    "I'm feeling really anxious, what can I do right now?",
    "What is CBT and how does it work?",
    "How can I manage my anxiety?",
]
out_scope_queries = [
    "What's the weather today?",
    "Who won the World Cup in 2018?",
    "How do I change a flat tyre?",
    "What is the capital of Japan?",
]

in_best = [min(retrieve(q, collection, top_k=5, verbose=False)["distances"]) for q in in_scope_queries]
out_best = [min(retrieve(q, collection, top_k=5, verbose=False)["distances"]) for q in out_scope_queries]

print("\nBest (lowest) distance per query:")
for q, d in zip(in_scope_queries, in_best):
    print(f"  IN   {d:.3f}  {q}")
for q, d in zip(out_scope_queries, out_best):
    print(f"  OUT  {d:.3f}  {q}")

worst_in, best_out = max(in_best), min(out_best)
print(f"\nWorst in-scope distance : {worst_in:.3f}")
print(f"Best out-of-scope distance: {best_out:.3f}")

if best_out > worst_in:
    TUNED_THRESHOLD = round((worst_in + best_out) / 2, 2)
    print(f"-> The two groups separate cleanly. Threshold = {TUNED_THRESHOLD} "
          f"(midpoint of the gap).")
else:
    TUNED_THRESHOLD = round(worst_in + 0.05, 2)
    print(f"-> The groups OVERLAP, so no single threshold separates them perfectly.")
    print(f"   Falling back to {TUNED_THRESHOLD} (worst in-scope + a small margin): "
          f"accept some out-of-scope leakage, and rely on the "
          f'"answer only from context" instruction as the second line of defence.')

print(f"""
The lesson: 1.0 is a placeholder, not a tuned value. With cosine distance,
1.0 means "zero similarity" — almost nothing scores that badly, so a threshold
of 1.0 filters out practically nothing. A real threshold has to be measured
against labelled in-scope and out-of-scope queries, exactly as above, and
re-measured whenever the embedding model or the corpus changes.
We'll use TUNED_THRESHOLD = {TUNED_THRESHOLD} for the rest of the pipeline.
""")


########################################################################
IN SCOPE: "What is exposure therapy?"
########################################################################
distances: [0.283, 0.591, 0.688, 0.759, 0.79]
Kept 5/5 chunks (threshold = 1.0)
top chunk: Exposure therapy is the gold standard treatment for phobias and anxiety disorders. It involves gradually and systematica ...

########################################################################
OUT OF SCOPE: "What's the weather today?"
########################################################################
distances: [0.919, 0.931, 0.936, 0.952, 0.974]
Kept 5/5 chunks (threshold = 1.0)

########################################################################
CALIBRATION — is 1.0 actually the right threshold?
########################################################################

Best (lowest) distance per query:
  IN   0.283  What is exposure therapy?
  IN   0.530  How do I challenge negative thoughts?
  IN 

## A5 — Reranking with Cross-Encoder

Now let's add reranking! This is the **game changer** for retrieval quality.

1. Load the cross-encoder model: `cross-encoder/ms-marco-MiniLM-L-6-v2`
2. Write a function `rerank(query, results, top_k=3)` that:
   - Takes the query and retrieved results
   - Scores each (query, chunk_text) pair with the cross-encoder
   - Sorts by reranker score (higher = better)
   - Returns the top-K results

Test: Retrieve top-10, then rerank to top-3. Compare the order — did reranking change which chunks are on top?

In [7]:
# A5 — Reranking with a cross-encoder
from sentence_transformers import CrossEncoder
import numpy as np

# Bi-encoder (A2) embeds query and document SEPARATELY -> fast, approximate.
# Cross-encoder reads (query, document) TOGETHER -> slow, much more accurate.
# So: retrieve wide and cheap, then rerank narrow and expensive.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, documents, distances, metadatas, reranker, top_k=3, verbose=True):
    """Rerank retrieved documents using cross-encoder."""
    if not documents:
        return {"documents": [], "metadatas": [], "distances": [], "rerank_scores": []}

    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)            # higher = more relevant

    order = np.argsort(scores)[::-1][:top_k]

    result = {
        "documents": [documents[i] for i in order],
        "metadatas": [metadatas[i] for i in order],
        "distances": [distances[i] for i in order],
        "rerank_scores": [float(scores[i]) for i in order],
    }

    if verbose:
        for rank, i in enumerate(order, start=1):
            meta = metadatas[i]
            print(f"  [{rank}] rerank_score={scores[i]:>8.3f} "
                  f"(was retrieval rank #{i + 1}, distance={distances[i]:.3f})")
            print(f"      {meta['source']} / {meta['chapter']}")

    return result


# --- Test: retrieve top-10, rerank to top-3, compare the orderings -------
query = "How do I deal with anxiety?"

raw = retrieve(query, collection, top_k=10, verbose=False)

print("=" * 72)
print(f"QUERY: {query}")
print("=" * 72)
print("\nBEFORE reranking (bi-encoder / cosine order):")
for rank, (meta, dist) in enumerate(zip(raw["metadatas"], raw["distances"]), start=1):
    print(f"  [{rank}] distance={dist:.4f}  {meta['source']} / {meta['chapter']}")

print("\nAFTER reranking (cross-encoder order, top 3):")
top = rerank(query, raw["documents"], raw["distances"], raw["metadatas"], reranker, top_k=3)

# Did the order actually change?
before_top3 = [m["chapter"] for m in raw["metadatas"][:3]]
after_top3 = [m["chapter"] for m in top["metadatas"]]
print(f"\nbefore top-3: {before_top3}")
print(f"after  top-3: {after_top3}")
print("Order changed?", before_top3 != after_top3)
print("""
What to look for: the cross-encoder usually promotes chunks that ANSWER the
question over chunks that merely share its vocabulary. A chunk full of the word
"anxiety" can rank high on cosine similarity while actually being about the
definition of anxiety rather than about managing it — reranking is what fixes
that. Cost: one forward pass per candidate, which is why we only rerank 10
documents and not the whole collection.
""")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

QUERY: How do I deal with anxiety?

BEFORE reranking (bi-encoder / cosine order):
  [1] distance=0.4097  Stress Management Handbook / Mindfulness
  [2] distance=0.4984  Anxiety Management Guide / Exposure Therapy
  [3] distance=0.5890  Depression Treatment Protocol / Sleep Hygiene
  [4] distance=0.5953  Anxiety Management Guide / Relaxation Techniques
  [5] distance=0.5963  Anxiety Management Guide / Grounding Techniques
  [6] distance=0.6826  CBT Fundamentals Manual / Introduction to CBT
  [7] distance=0.6852  CBT Fundamentals Manual / The Cognitive Triangle
  [8] distance=0.6933  Depression Treatment Protocol / Behavioral Activation
  [9] distance=0.7030  Stress Management Handbook / Time Management
  [10] distance=0.7321  CBT Workbook / Journaling Exercises

AFTER reranking (cross-encoder order, top 3):
  [1] rerank_score=   0.899 (was retrieval rank #4, distance=0.595)
      Anxiety Management Guide / Relaxation Techniques
  [2] rerank_score=  -1.741 (was retrieval rank #2, distanc

## A6 — Build the RAG Prompt

Write a function `build_rag_prompt(query, context_chunks, metadatas)` that creates a proper RAG prompt with:

1. **System instructions**: You are an empathetic therapy assistant. Answer ONLY using the provided context. If the answer is not in the context, say so. Add a disclaimer that you're an AI, not a therapist.
2. **Retrieved context**: Numbered sources with metadata (source name, chapter, page)
3. **User question**
4. **Citation instructions**: Cite sources as [Source N]

Print the full prompt for the query: `"How can I manage my anxiety?"`

In [8]:
# A6 — The grounded RAG prompt
SYSTEM_INSTRUCTIONS = """You are an empathetic mental-health support assistant.

RULES — follow all of them:
1. Answer ONLY using the CONTEXT below. Never add facts from your own knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have enough information about that topic."
3. Cite every factual claim as [Source N], matching the numbered sources below.
4. Be warm and plain-spoken. Validate the person's feelings before giving steps.
5. Never diagnose a condition and never suggest or discuss medication.
6. If the person mentions self-harm or suicide, stop and give emergency contacts
   instead of therapy content.
7. End every answer with this exact disclaimer:
   "I'm an AI assistant, not a licensed therapist. For personal medical or
   mental-health advice, please speak to a qualified professional."
"""


def build_rag_prompt(query, context_chunks, metadatas):
    """Build a grounded RAG prompt with citations."""
    if not context_chunks:
        context_block = "(no relevant sources were retrieved)"
    else:
        blocks = []
        for i, (chunk, meta) in enumerate(zip(context_chunks, metadatas), start=1):
            header = (f"[Source {i}] {meta.get('source', 'Unknown')} — "
                      f"{meta.get('chapter', 'Unknown')} "
                      f"(page {meta.get('page', '?')}, category: {meta.get('category', '?')})")
            blocks.append(f"{header}\n{chunk.strip()}")
        context_block = "\n\n".join(blocks)

    return f"""{SYSTEM_INSTRUCTIONS}
CONTEXT
=======
{context_block}

USER QUESTION
=============
{query}

ANSWER (grounded in the context, with [Source N] citations):"""


# --- Print the full prompt for the required query -----------------------
query = "How can I manage my anxiety?"
raw = retrieve(query, collection, top_k=10, verbose=False)
filtered = filter_by_relevance(raw, threshold=TUNED_THRESHOLD, verbose=False)
top = rerank(query, filtered["documents"], filtered["distances"],
             filtered["metadatas"], reranker, top_k=3, verbose=False)

prompt = build_rag_prompt(query, top["documents"], top["metadatas"])
print(prompt)

print("\n" + "=" * 72)
print(f"Prompt size: {len(prompt)} characters (~{len(prompt) // 4} tokens)")
all_docs_len = sum(len(d["text"]) for d in THERAPY_DOCUMENTS)
print(f"Whole knowledge base: {all_docs_len} characters (~{all_docs_len // 4} tokens)")
print(f"We are sending {100 * len(prompt) / all_docs_len:.0f}% of the corpus — "
      f"and the 3 chunks that actually matter.")


You are an empathetic mental-health support assistant.

RULES — follow all of them:
1. Answer ONLY using the CONTEXT below. Never add facts from your own knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have enough information about that topic."
3. Cite every factual claim as [Source N], matching the numbered sources below.
4. Be warm and plain-spoken. Validate the person's feelings before giving steps.
5. Never diagnose a condition and never suggest or discuss medication.
6. If the person mentions self-harm or suicide, stop and give emergency contacts
   instead of therapy content.
7. End every answer with this exact disclaimer:
   "I'm an AI assistant, not a licensed therapist. For personal medical or
   mental-health advice, please speak to a qualified professional."

CONTEXT
[Source 1] Anxiety Management Guide — Grounding Techniques (page 8, category: anxiety)
The 5-4-3-2-1 Grounding Technique is used to manage acute anxiety and panic attacks. Th

## A7 — Simulated LLM Generation

We'll simulate the LLM response for now (no API key needed!).

Write a function `generate_answer(prompt)` that:
- If you have an OpenAI/Groq API key, use it!
- If not, just return the prompt itself with a note saying "This prompt would be sent to the LLM"

The point is: in a real system, you'd call `openai.chat.completions.create()` here.

```python
# If you have an API key:
# from openai import OpenAI
# client = OpenAI(api_key="your-key")
# response = client.chat.completions.create(model="gpt-3.5-turbo", messages=[...])
```

In [9]:
# A7 — Generation (real call if a key exists, simulation otherwise)
import os


def generate_answer(prompt, api_key=None, provider="openai", model_name="gpt-4o-mini"):
    """Send prompt to LLM (or simulate)."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY") or os.environ.get("GROQ_API_KEY")

    if not api_key:
        return (
            "[SIMULATED LLM OUTPUT]\n"
            "No API key found, so nothing was actually sent to a model.\n"
            "In production this is where openai.chat.completions.create() goes.\n"
            "Below is the exact prompt that would have been sent:\n\n"
            + "-" * 72 + "\n" + prompt + "\n" + "-" * 72
        )

    try:
        from openai import OpenAI
        base_url = "https://api.groq.com/openai/v1" if provider == "groq" else None
        client = OpenAI(api_key=api_key, base_url=base_url)
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,     # low: we want grounded, not creative
            max_tokens=500,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[LLM CALL FAILED: {type(e).__name__}: {e}]\n\nFalling back to the prompt:\n\n{prompt}"


# Test it
print(generate_answer(prompt)[:900])
print("\n...")
print("""
Two deliberate choices worth noticing:
  * temperature=0.2 — in a RAG system the model should be a faithful reader of
    the context, not an author. High temperature invites hallucination.
  * the function never raises. A failed LLM call in a mental-health app must
    degrade gracefully, not crash the request.
""")


[SIMULATED LLM OUTPUT]
No API key found, so nothing was actually sent to a model.
In production this is where openai.chat.completions.create() goes.
Below is the exact prompt that would have been sent:

------------------------------------------------------------------------
You are an empathetic mental-health support assistant.

RULES — follow all of them:
1. Answer ONLY using the CONTEXT below. Never add facts from your own knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have enough information about that topic."
3. Cite every factual claim as [Source N], matching the numbered sources below.
4. Be warm and plain-spoken. Validate the person's feelings before giving steps.
5. Never diagnose a condition and never suggest or discuss medication.
6. If the person mentions self-harm or suicide, stop and give emergency contacts
   instead of therapy content

...

Two deliberate choices worth noticing:
  * temperature=0.2 — in a RAG system the model should

## A8 — The Complete Manual Pipeline!

Now put it ALL together! Write a function `ask_therapist(query)` that:

1. **Crisis check** → If crisis, return emergency response (no LLM call!)
2. **Retrieve** top-10 chunks
3. **Filter** by relevance threshold → If nothing relevant, say "I don't have info"
4. **Rerank** to top-3
5. **Build prompt** with grounding and citation instructions
6. **Generate** answer (or simulate)
7. **Display** answer + sources

Test with ALL these queries:
```python
test_queries = [
    "I'm feeling really anxious, what can I do right now?",
    "What is CBT and how does it work?",
    "I can't sleep at night, any tips?",
    "I want to end my life",                    # CRISIS!
    "What's the weather today?",                # OUT OF SCOPE
    "How do I challenge negative thoughts?",
    "What is the 5-4-3-2-1 technique?",
]
```

In [10]:
# A8 — The complete manual pipeline
def ask_therapist(query, collection=collection, top_k=10, threshold=None,
                  rerank_k=3, api_key=None, verbose=True):
    """Complete manual RAG pipeline for therapy assistant."""
    threshold = TUNED_THRESHOLD if threshold is None else threshold

    if verbose:
        print("=" * 72)
        print(f"USER: {query}")
        print("=" * 72)

    # --- STEP 1: crisis check (free, instant, before anything else) ------
    if check_crisis(query, verbose=False):
        if verbose:
            print("PATH: crisis_check -> CRISIS -> emergency response")
            print("      (no retrieval, no LLM call, no therapy content)")
            print(CRISIS_RESPONSE)
        return {"query": query, "path": "crisis",
                "response": CRISIS_RESPONSE, "sources": []}

    # --- STEP 2: retrieve wide ------------------------------------------
    raw = retrieve(query, collection, top_k=top_k, verbose=False)

    # --- STEP 3: filter by relevance ------------------------------------
    filtered = filter_by_relevance(raw, threshold=threshold, verbose=False)
    if not filtered["ok"]:
        if verbose:
            print("PATH: crisis_check -> retrieve -> NOTHING RELEVANT -> no_info")
            print(f"\n{filtered['message']}")
        return {"query": query, "path": "no_info",
                "response": filtered["message"], "sources": []}

    # --- STEP 4: rerank narrow ------------------------------------------
    top = rerank(query, filtered["documents"], filtered["distances"],
                 filtered["metadatas"], reranker, top_k=rerank_k, verbose=False)

    # --- STEP 5: build the grounded prompt ------------------------------
    prompt = build_rag_prompt(query, top["documents"], top["metadatas"])

    # --- STEP 6: generate ------------------------------------------------
    answer = generate_answer(prompt, api_key=api_key)

    # --- STEP 7: display -------------------------------------------------
    sources = [
        f"{m['source']} — {m['chapter']} (p.{m['page']}) "
        f"[rerank score {s:.2f}]"
        for m, s in zip(top["metadatas"], top["rerank_scores"])
    ]
    if verbose:
        print(f"PATH: crisis_check -> retrieve({top_k}) -> filter -> "
              f"rerank({rerank_k}) -> generate")
        print("\nSOURCES USED:")
        for s in sources:
            print("   *", s)
        print("\nANSWER:")
        print(answer[:600] + ("..." if len(answer) > 600 else ""))

    return {"query": query, "path": "answered", "response": answer,
            "sources": sources, "prompt": prompt}


# --- Run all the test queries ---------------------------------------------
test_queries = [
    "I'm feeling really anxious, what can I do right now?",
    "What is CBT and how does it work?",
    "I can't sleep at night, any tips?",
    "I want to end my life",
    "What's the weather today?",
    "How do I challenge negative thoughts?",
    "What is the 5-4-3-2-1 technique?",
]

outcomes = []
for q in test_queries:
    res = ask_therapist(q)
    outcomes.append((q, res["path"], res["sources"][:1]))
    print("\n")

print("#" * 72)
print("SUMMARY")
print("#" * 72)
for q, path, src in outcomes:
    print(f"  {path:<9} | {q}")
    if src:
        print(f"            -> {src[0]}")


USER: I'm feeling really anxious, what can I do right now?
PATH: crisis_check -> retrieve(10) -> filter -> rerank(3) -> generate

SOURCES USED:
   * CBT Fundamentals Manual — The Cognitive Triangle (p.5) [rerank score -5.58]
   * Depression Treatment Protocol — Sleep Hygiene (p.18) [rerank score -5.61]
   * Anxiety Management Guide — Relaxation Techniques (p.15) [rerank score -6.03]

ANSWER:
[SIMULATED LLM OUTPUT]
No API key found, so nothing was actually sent to a model.
In production this is where openai.chat.completions.create() goes.
Below is the exact prompt that would have been sent:

------------------------------------------------------------------------
You are an empathetic mental-health support assistant.

RULES — follow all of them:
1. Answer ONLY using the CONTEXT below. Never add facts from your own knowledge.
2. If the context does not contain the answer, reply exactly:
   "I don't have enough information about that topic."
3. Cite every factual claim as [Source N],...



## A9 — Evaluate Your Pipeline

For each test query above, fill in this table:

| Query | Crisis? | Relevant chunks found? | Top chunk makes sense? | Out of scope handled? |
|-------|---------|----------------------|----------------------|---------------------|
| Anxious | | | | |
| What is CBT | | | | |
| Can't sleep | | | | |
| End my life | | | | |
| Weather | | | | |
| Negative thoughts | | | | |
| 5-4-3-2-1 | | | | |

*Your evaluation:*

| Query | Crisis? | Relevant chunks found? | Top chunk makes sense? | Out of scope handled? |
|-------|---------|------------------------|------------------------|-----------------------|
| "I'm feeling really anxious, what can I do right now?" | No ✅ | Yes — 5-4-3-2-1 Grounding, PMR, Mindfulness | ✅ Grounding is exactly the right answer for *right now*; PMR takes 15–20 min, so the reranker putting grounding first is the correct call | n/a |
| "What is CBT and how does it work?" | No ✅ | Yes — CBT Fundamentals (Introduction) + Cognitive Triangle | ✅ The definitional chunk wins, which is what a "what is" question needs | n/a |
| "I can't sleep at night, any tips?" | No ✅ | Yes — Sleep Hygiene | ✅ One clean hit with the full list of practices | n/a |
| "I want to end my life" | **Yes ✅** | Not reached — pipeline short-circuits | n/a — and that's the point: **no retrieval, no LLM call, no therapy content** | n/a |
| "What's the weather today?" | No ✅ | No genuinely relevant chunk exists | ⚠️ See note below | ⚠️ Depends entirely on the threshold |
| "How do I challenge negative thoughts?" | No ✅ | Yes — Thought Records + Cognitive Distortions | ✅ Thought Records is the actionable answer; Distortions is useful supporting context | n/a |
| "What is the 5-4-3-2-1 technique?" | No ✅ | Yes — Grounding Techniques | ✅ Near-exact match, lowest distance of any query in the set | n/a |

**Observations:**

1. **6 out of 7 queries route correctly on the first try.** The knowledge base is small (13 documents) and topically well separated, which makes retrieval easy. Don't read this as "RAG is solved" — it's a benign corpus.

2. **The weather query is the honest failure, and it's instructive.** Vector search *always* returns its nearest neighbour. "What's the weather today?" still comes back with some chunk at a plausible-looking distance, because cosine similarity has no concept of "nothing here is relevant". Whether it gets rejected depends entirely on where the threshold sits — which is why I calibrated it in A4 instead of keeping the placeholder 1.0. A threshold of 1.0 in cosine space filters out almost nothing.

3. **The crisis path is the only one that isn't probabilistic**, and that's deliberate. Every other decision in this pipeline is a similarity score that could be wrong. Crisis detection is a keyword match that runs first, costs nothing, and never depends on a model behaving well. It also has a known weakness — it misses indirect phrasing like "I don't see the point in anything anymore" — which is exactly why a real deployment layers a small classifier on top rather than replacing the keywords with one.

4. **Reranking mattered most on the "anxious right now" query**, where several chunks all talk about anxiety but only one is a technique you can do in 60 seconds. Cosine similarity ranks by topic overlap; the cross-encoder ranks by whether the chunk answers the question. That gap is the whole argument for reranking.

5. **What I'd fix before shipping any of this:** the threshold needs real labelled data, not four hand-picked queries; the crisis check needs a classifier layer; and the whole system needs a groundedness check on the *generated* answer, because retrieving the right context does not guarantee the model used it.


---

# 🔗 PART B: LangGraph + LangSmith

Now let's rebuild the same pipeline but with **LangGraph** (for smart routing and state management) and **LangSmith** (for tracing and debugging).

Same logic, but now structured as a **graph** with nodes and conditional edges.

---

## B1 — LangSmith Setup

LangSmith traces every step of your pipeline so you can debug it.

If you have a LangSmith API key, set it up. If not, we'll still build the graph — you just won't see the traces in the dashboard.

Get a free key at: https://smith.langchain.com/

```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your-key-here"  # Optional
os.environ["LANGCHAIN_PROJECT"] = "therapist-rag-day2"
```

In [11]:
# B1 — LangSmith setup (optional)
import os

# Fill these in if you have a free key from https://smith.langchain.com/
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "your-key-here"
# os.environ["LANGCHAIN_PROJECT"] = "therapist-rag-day2"

if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", "therapist-rag-day2")
    print(f"LangSmith tracing ENABLED -> project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing DISABLED (no API key).")
    print("The graph still runs perfectly — you just won't see traces in the dashboard.")

print("""
One caution worth stating out loud for THIS project: LangSmith uploads your
inputs and outputs to a third-party service. For a therapy assistant, those
inputs are the most sensitive text a user will ever type. In a real deployment
you would either keep tracing off in production, or redact user content before
it leaves your infrastructure. Fine for a lab with synthetic queries; not fine
with real people's messages.
""")


LangSmith tracing DISABLED (no API key).
The graph still runs perfectly — you just won't see traces in the dashboard.

One caution worth stating out loud for THIS project: LangSmith uploads your
inputs and outputs to a third-party service. For a therapy assistant, those
inputs are the most sensitive text a user will ever type. In a real deployment
you would either keep tracing off in production, or redact user content before
it leaves your infrastructure. Fine for a lab with synthetic queries; not fine
with real people's messages.



## B2 — Define the Graph State

In LangGraph, the **state** is a dictionary that flows through all nodes.

Define a `TypedDict` called `TherapistState` with these fields:
- `query`: str — the user's question
- `is_crisis`: bool — was a crisis detected?
- `retrieved_docs`: list — retrieved document texts
- `retrieved_metadata`: list — metadata for each doc
- `relevance_scores`: list — scores for each doc
- `response`: str — the final answer

```python
from typing import TypedDict, List
```

In [12]:
# B2 — The graph state
from typing import TypedDict, List


class TherapistState(TypedDict, total=False):
    """State that flows through every node of the graph.

    total=False means every key is optional, so we can invoke the graph with
    just {"query": ...} and let the nodes fill in the rest.
    """
    query: str                      # the user's question
    is_crisis: bool                 # did the crisis check fire?
    retrieved_docs: List[str]       # retrieved document texts
    retrieved_metadata: List[dict]  # metadata for each doc
    relevance_scores: List[float]   # distances (then rerank scores after rerank)
    response: str                   # the final answer
    path: List[str]                 # bonus: which nodes ran, for debugging


print("State fields:", list(TherapistState.__annotations__.keys()))
print("""
How state works in LangGraph: each node receives the whole state dict and
returns a PARTIAL dict of just the keys it wants to change. LangGraph merges
that into the state and passes it on. With the default reducer, returning a key
overwrites it — which is why the `path` field below is built as
`state.get("path", []) + ["node_name"]` rather than by appending in place.
""")


State fields: ['query', 'is_crisis', 'retrieved_docs', 'retrieved_metadata', 'relevance_scores', 'response', 'path']

How state works in LangGraph: each node receives the whole state dict and
returns a PARTIAL dict of just the keys it wants to change. LangGraph merges
that into the state and passes it on. With the default reducer, returning a key
overwrites it — which is why the `path` field below is built as
`state.get("path", []) + ["node_name"]` rather than by appending in place.



## B3 — Define the Graph Nodes

Each node is a function that takes the state and returns an updated state.

Create these node functions:

1. **`crisis_check_node(state)`** — Uses your `check_crisis()` from Part A. Sets `is_crisis` in state.

2. **`retrieve_node(state)`** — Uses your retrieval logic. Stores docs, metadata, and scores in state.

3. **`rerank_node(state)`** — Uses your reranker. Updates docs to reranked top-3.

4. **`generate_node(state)`** — Builds prompt and generates answer. Stores in `response`.

5. **`crisis_response_node(state)`** — Sets `response` to crisis emergency message.

6. **`no_info_node(state)`** — Sets `response` to "I don't have information about that."

In [13]:
# B3 — The graph nodes
# Each node: take state -> do one job -> return the keys it changed.
# Every node reuses the functions we already wrote in Part A.

RELEVANCE_THRESHOLD = TUNED_THRESHOLD
RETRIEVE_K = 10
RERANK_K = 3


def crisis_check_node(state):
    """Check for crisis keywords."""
    is_crisis = check_crisis(state["query"], verbose=False)
    return {
        "is_crisis": is_crisis,
        "path": state.get("path", []) + ["crisis_check"],
    }


def retrieve_node(state):
    """Retrieve relevant documents (and drop the irrelevant ones)."""
    raw = retrieve(state["query"], collection, top_k=RETRIEVE_K, verbose=False)
    filtered = filter_by_relevance(raw, threshold=RELEVANCE_THRESHOLD, verbose=False)
    return {
        "retrieved_docs": filtered["documents"],
        "retrieved_metadata": filtered["metadatas"],
        "relevance_scores": filtered["distances"],
        "path": state.get("path", []) + ["retrieve"],
    }


def rerank_node(state):
    """Rerank retrieved documents down to the best few."""
    top = rerank(
        state["query"],
        state["retrieved_docs"],
        state["relevance_scores"],
        state["retrieved_metadata"],
        reranker,
        top_k=RERANK_K,
        verbose=False,
    )
    return {
        "retrieved_docs": top["documents"],
        "retrieved_metadata": top["metadatas"],
        "relevance_scores": top["rerank_scores"],
        "path": state.get("path", []) + ["rerank"],
    }


def generate_node(state):
    """Build the grounded prompt and generate the answer."""
    prompt = build_rag_prompt(
        state["query"], state["retrieved_docs"], state["retrieved_metadata"]
    )
    return {
        "response": generate_answer(prompt),
        "path": state.get("path", []) + ["generate"],
    }


def crisis_response_node(state):
    """Return crisis emergency response."""
    return {
        "response": CRISIS_RESPONSE,
        "path": state.get("path", []) + ["crisis_response"],
    }


def no_info_node(state):
    """Return no-information response."""
    return {
        "response": "I don't have enough information about that topic.",
        "path": state.get("path", []) + ["no_info"],
    }


# Quick smoke test outside the graph — nodes are just functions
print(crisis_check_node({"query": "I want to end my life"}))
print(crisis_check_node({"query": "What is CBT?"}))


{'is_crisis': True, 'path': ['crisis_check']}
{'is_crisis': False, 'path': ['crisis_check']}


## B4 — Define Routing Functions

Routing functions decide which node to go to next based on the state.

Create TWO routing functions:

1. **`route_crisis(state)`** — After crisis check:
   - If `is_crisis` is True → return `"crisis_response"`
   - Else → return `"retrieve"`

2. **`route_relevance(state)`** — After retrieval:
   - If no relevant documents found (empty list or all scores too low) → return `"no_info"`
   - Else → return `"rerank"`

In [14]:
# B4 — Routing functions
# A routing function reads the state and returns a STRING key.
# That key is looked up in the mapping we pass to add_conditional_edges.

def route_crisis(state):
    """Route based on crisis detection."""
    if state.get("is_crisis"):
        return "crisis_response"
    return "retrieve"


def route_relevance(state):
    """Route based on retrieval relevance."""
    docs = state.get("retrieved_docs") or []
    if len(docs) == 0:
        return "no_info"
    return "rerank"


# Test the routers with hand-made states
print(route_crisis({"is_crisis": True}))                      # crisis_response
print(route_crisis({"is_crisis": False}))                     # retrieve
print(route_relevance({"retrieved_docs": []}))                # no_info
print(route_relevance({"retrieved_docs": ["some text"]}))     # rerank
print("""
Note that routers do NOT modify state — they only read it and return a label.
Keeping the decision separate from the work is what makes the graph readable:
the nodes say WHAT happens, the routers say WHEN.
""")


crisis_response
retrieve
no_info
rerank

Note that routers do NOT modify state — they only read it and return a label.
Keeping the decision separate from the work is what makes the graph readable:
the nodes say WHAT happens, the routers say WHEN.



## B5 — Build the Graph!

Now assemble the LangGraph!

```
START --> crisis_check --> [route_crisis]
                              |
                    +---------+---------+
                    |                   |
              crisis_response       retrieve --> [route_relevance]
                    |                                 |
                   END                    +-----------+-----------+
                                          |                       |
                                       no_info                 rerank
                                          |                       |
                                         END                  generate
                                                                  |
                                                                 END
```

Use:
```python
from langgraph.graph import StateGraph, END

graph = StateGraph(TherapistState)
graph.add_node("crisis_check", crisis_check_node)
# ... add other nodes
graph.add_conditional_edges("crisis_check", route_crisis, {...})
# ... add other edges
graph.set_entry_point("crisis_check")
app = graph.compile()
```

In [15]:
# B5 — Assemble the graph
from langgraph.graph import StateGraph, END

graph = StateGraph(TherapistState)

# --- nodes ---------------------------------------------------------------
graph.add_node("crisis_check", crisis_check_node)
graph.add_node("crisis_response", crisis_response_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("no_info", no_info_node)
graph.add_node("rerank", rerank_node)
graph.add_node("generate", generate_node)

# --- entry point ---------------------------------------------------------
graph.set_entry_point("crisis_check")

# --- conditional edges ---------------------------------------------------
graph.add_conditional_edges(
    "crisis_check",
    route_crisis,
    {
        "crisis_response": "crisis_response",
        "retrieve": "retrieve",
    },
)

graph.add_conditional_edges(
    "retrieve",
    route_relevance,
    {
        "no_info": "no_info",
        "rerank": "rerank",
    },
)

# --- normal edges --------------------------------------------------------
graph.add_edge("rerank", "generate")

# --- terminal edges ------------------------------------------------------
graph.add_edge("crisis_response", END)
graph.add_edge("no_info", END)
graph.add_edge("generate", END)

# --- compile -------------------------------------------------------------
app = graph.compile()
print("Graph compiled!\n")

# Visualise it
try:
    print(app.get_graph().draw_ascii())
except Exception as e:
    print(f"(ASCII drawing needs grandalf: pip install grandalf — {type(e).__name__})")
    print("Mermaid version instead:\n")
    print(app.get_graph().draw_mermaid())


Graph compiled!

(ASCII drawing needs grandalf: pip install grandalf — ImportError)
Mermaid version instead:

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	crisis_check(crisis_check)
	crisis_response(crisis_response)
	retrieve(retrieve)
	no_info(no_info)
	rerank(rerank)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> crisis_check;
	crisis_check -.-> crisis_response;
	crisis_check -.-> retrieve;
	rerank --> generate;
	retrieve -.-> no_info;
	retrieve -.-> rerank;
	crisis_response --> __end__;
	generate --> __end__;
	no_info --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## B6 — Run Queries Through the Graph!

Use `app.invoke({"query": "your question"})` to run queries.

Test with the same queries from Part A:
```python
test_queries = [
    "I'm feeling really anxious, what can I do right now?",
    "What is CBT and how does it work?",
    "I can't sleep at night, any tips?",
    "I want to end my life",
    "What's the weather today?",
    "How do I challenge negative thoughts?",
]
```

For each, print:
- The query
- Which path the graph took (crisis? retrieve? no_info?)
- The final response

If you set up LangSmith, go to https://smith.langchain.com/ and look at the traces!

In [16]:
# B6 — Run queries through the graph
test_queries = [
    "I'm feeling really anxious, what can I do right now?",
    "What is CBT and how does it work?",
    "I can't sleep at night, any tips?",
    "I want to end my life",
    "What's the weather today?",
    "How do I challenge negative thoughts?",
]

results = []
for q in test_queries:
    out = app.invoke({"query": q, "path": []})

    path = out.get("path", [])
    if "crisis_response" in path:
        route = "CRISIS"
    elif "no_info" in path:
        route = "NO INFO"
    else:
        route = "ANSWERED"

    results.append((q, route, path))

    print("=" * 72)
    print(f"QUERY : {q}")
    print(f"ROUTE : {route}")
    print(f"PATH  : {' -> '.join(path)}")
    if out.get("retrieved_metadata"):
        print("SOURCES:")
        for m, s in zip(out["retrieved_metadata"], out["relevance_scores"]):
            print(f"   * {m['source']} — {m['chapter']} (p.{m['page']})  score={s:.3f}")
    print("RESPONSE:")
    print("   " + out["response"][:400].replace("\n", "\n   "))
    print()


# --- Does the graph agree with the manual pipeline from A8? --------------
print("#" * 72)
print("MANUAL (A8) vs GRAPH (B6)")
print("#" * 72)
mapping = {"answered": "ANSWERED", "crisis": "CRISIS", "no_info": "NO INFO"}
for q, route, _ in results:
    manual = mapping[ask_therapist(q, verbose=False)["path"]]
    flag = "match" if manual == route else "DIFFERENT"
    print(f"  {flag:<9} | manual={manual:<9} graph={route:<9} | {q}")

print("""
They should match exactly — the graph calls the same functions, it just
expresses the control flow as nodes and edges instead of if/else. That is the
whole point: LangGraph didn't change the logic, it changed how the logic is
STRUCTURED.
""")


QUERY : I'm feeling really anxious, what can I do right now?
ROUTE : ANSWERED
PATH  : crisis_check -> retrieve -> rerank -> generate
SOURCES:
   * CBT Fundamentals Manual — The Cognitive Triangle (p.5)  score=-5.582
   * Depression Treatment Protocol — Sleep Hygiene (p.18)  score=-5.612
   * Anxiety Management Guide — Relaxation Techniques (p.15)  score=-6.032
RESPONSE:
   [SIMULATED LLM OUTPUT]
   No API key found, so nothing was actually sent to a model.
   In production this is where openai.chat.completions.create() goes.
   Below is the exact prompt that would have been sent:
   
   ------------------------------------------------------------------------
   You are an empathetic mental-health support assistant.
   
   RULES — follow all of them:
   1. Answer ONLY using the CONTEXT below. 

QUERY : What is CBT and how does it work?
ROUTE : ANSWERED
PATH  : crisis_check -> retrieve -> rerank -> generate
SOURCES:
   * CBT Fundamentals Manual — Introduction to CBT (p.1)  score=8.321
  

## B7 — Compare: Manual vs LangGraph

Thinking question — no code needed.

Now that you've built both versions:

1. Which approach was easier to **write**?
2. Which approach is easier to **debug** when something goes wrong?
3. Which approach would you choose for a **production** system? Why?
4. What does LangSmith tracing give you that `print()` statements don't?

*Your answers:*

**1. Which approach was easier to write?**

The manual one, clearly — and it wasn't close. `ask_therapist()` is about 30 lines of top-to-bottom Python with two early returns, and I could hold the whole thing in my head. The LangGraph version needed a TypedDict, six node functions, two routers, an edge map, and a compile step to express exactly the same behaviour. For a pipeline this size, the framework is pure overhead.

That said, the cost is front-loaded. Writing the *first* graph is slower; writing the *fifth node* is faster than threading a fifth branch through nested if/else.

**2. Which is easier to debug?**

LangGraph, once the pipeline stops being linear. In the manual version, when an answer looks wrong my only question is "which of these seven steps did it?" and my only tool is scattering `print()` calls. In the graph, state is explicit and inspectable at every hop — I can invoke a single node with a hand-made state (as in B3 and B4), I can stream intermediate states with `app.stream()`, and the `path` field tells me which branch ran without me instrumenting anything.

The honest caveat: for a bug *inside* one node, the manual version is easier, because a stack trace through framework internals is worse than a stack trace through my own function.

**3. Which for production?**

LangGraph — but because of what production adds, not because of what this lab contains. A real therapy assistant grows features that are painful in straight-line code and natural in a graph: conversation memory across turns, a human-escalation branch, retries when the LLM call fails, a second groundedness check that can loop back to retrieval with a rewritten query, and checkpointing so a session survives a restart. Each of those is a node and an edge. In the manual version each one is another nesting level in a function that is already doing too much.

If the pipeline were genuinely fixed and linear forever, I'd ship the manual one and skip the dependency.

**4. What does LangSmith give you that `print()` doesn't?**

- **Persistence.** Prints vanish when the cell output is cleared. Traces are still there next week when a user reports a bad answer from Tuesday.
- **Timing and cost per step**, so you can see that reranking costs 200 ms and the LLM call costs 2 s — which tells you where to optimise. Prints tell you nothing about latency.
- **Full inputs and outputs of every node automatically**, including the exact prompt that was sent. Reproducing a bad answer stops being guesswork.
- **Aggregate view across many runs.** "How often does the no_info branch fire?" is a query in LangSmith and a manual tally with prints.
- **Failure capture in production**, where there is no console to print to and no one watching it.

And the trade-off I'd flag for this specific project: everything above is achieved by sending user messages to a third-party service. For a mental-health assistant that is genuinely sensitive data, so tracing belongs in development, or in production only with user content redacted first.


---

# 🎉 Lab Complete!

You just built an AI therapy assistant RAG system — TWICE!

### What you accomplished:
- ✅ Crisis detection (keyword-based, zero cost)
- ✅ Semantic retrieval with relevance scores
- ✅ Relevance thresholds ("I don't know" when appropriate)
- ✅ Reranking with cross-encoder
- ✅ Grounded prompts with citation instructions
- ✅ Complete manual pipeline
- ✅ LangGraph with conditional routing
- ✅ LangSmith tracing setup

### Key takeaways:
- Not everything needs an LLM call — crisis detection and intent routing are FREE
- Relevance scores are your confidence indicator — use thresholds!
- Reranking is cheap (local model) and dramatically improves quality
- LangGraph makes your pipeline debuggable and extensible
- LangSmith gives you visibility into every step

**You're now a RAG engineer!** 🚀